# Protocol Redundancy Analysis (PRA)
## MedBIND3D - Analyzing Multimodal MRI Redundancy

**Objective**: Quantify information redundancy between MRI modalities (FLAIR, T1, T1CE, T2) using:
- **DCCA** (Deep Canonical Correlation Analysis) - Nonlinear shared representations
- **HSIC** (Hilbert-Schmidt Independence Criterion) - Kernel-based dependence scores

**Dataset**: BraTS2020 Validation Set (no segmentation masks)

**Output**: Redundancy metrics + 5 advanced visualizations

---
## 1. Import Libraries and Setup

In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import nibabel as nib
import pandas as pd
from tqdm import tqdm
from itertools import combinations

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.patches import FancyBboxPatch, Circle
from matplotlib.collections import LineCollection
import matplotlib.patches as mpatches

# Seaborn styling for research publications
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.2)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

---
## 2. Configuration and Paths

In [ ]:
# Project structure
PROJECT_ROOT = Path('..')
DATA_ROOT = PROJECT_ROOT / 'BraTS2020_ValidationData' / 'MICCAI_BraTS2020_ValidationData'
OUTPUT_DIR = Path('/home/claude/MLFM/results')
VIZ_DIR = OUTPUT_DIR / 'PRA_visualizations'
SCORES_DIR = OUTPUT_DIR / 'pra_scores'

# Create directories
VIZ_DIR.mkdir(parents=True, exist_ok=True)
SCORES_DIR.mkdir(parents=True, exist_ok=True)

# Modalities
MODALITIES = ['flair', 't1', 't1ce', 't2']
MODALITY_NAMES = {'flair': 'FLAIR', 't1': 'T1', 't1ce': 'T1CE', 't2': 'T2'}
MODALITY_PAIRS = list(combinations(MODALITIES, 2))

# Hyperparameters
PATCH_SIZE = 64
LATENT_DIM = 32
BATCH_SIZE = 8
NUM_EPOCHS_DCCA = 20
LEARNING_RATE = 1e-4

print(f"Data Root: {DATA_ROOT.resolve()}")
print(f"Output Dir: {OUTPUT_DIR.resolve()}")
print(f"\nModality Pairs: {len(MODALITY_PAIRS)}")
for pair in MODALITY_PAIRS:
    print(f"  {MODALITY_NAMES[pair[0]]} <-> {MODALITY_NAMES[pair[1]]}")

---
## 3. Data Loading and Preprocessing

In [ ]:
class BraTS2020MultimodalDataset(Dataset):
    """BraTS2020 Dataset for multimodal redundancy analysis"""
    
    def __init__(self, data_root, modalities, patch_size=64, num_patches_per_volume=5):
        self.data_root = Path(data_root)
        self.modalities = modalities
        self.patch_size = patch_size
        self.num_patches_per_volume = num_patches_per_volume
        
        # Find all patient directories
        self.patient_dirs = sorted([d for d in self.data_root.iterdir() if d.is_dir()])
        print(f"Found {len(self.patient_dirs)} patients")
        
        # Pre-extract patches
        self.patches = {mod: [] for mod in modalities}
        self.patient_ids = []
        
        print("\nLoading and preprocessing volumes...")
        for patient_dir in tqdm(self.patient_dirs[:30], desc="Loading patients"):  # Use first 30
            patient_id = patient_dir.name
            
            # Load all modalities
            volumes = {}
            for mod in self.modalities:
                file_path = patient_dir / f"{patient_id}_{mod}.nii"
                if file_path.exists():
                    vol = nib.load(str(file_path)).get_fdata()
                    # Z-score normalization
                    vol = (vol - vol.mean()) / (vol.std() + 1e-8)
                    volumes[mod] = vol
            
            if len(volumes) == len(self.modalities):
                # Extract random patches
                vol_shape = list(volumes.values())[0].shape
                
                for _ in range(self.num_patches_per_volume):
                    # Random location
                    x = np.random.randint(20, vol_shape[0] - self.patch_size - 20)
                    y = np.random.randint(20, vol_shape[1] - self.patch_size - 20)
                    z = np.random.randint(20, vol_shape[2] - self.patch_size - 20)
                    
                    # Extract patches from all modalities
                    patches_ok = True
                    temp_patches = {}
                    for mod in self.modalities:
                        patch = volumes[mod][x:x+self.patch_size, 
                                            y:y+self.patch_size, 
                                            z:z+self.patch_size]
                        if patch.std() > 0.1:  # Skip background
                            temp_patches[mod] = patch.copy()
                        else:
                            patches_ok = False
                            break
                    
                    if patches_ok:
                        for mod in self.modalities:
                            self.patches[mod].append(temp_patches[mod])
                        self.patient_ids.append(patient_id)
        
        # Convert to arrays
        for mod in self.modalities:
            self.patches[mod] = np.array(self.patches[mod])
        
        print(f"\nTotal patches extracted: {len(self.patient_ids)}")
        print(f"Patch shape: {self.patches[self.modalities[0]][0].shape}")
        
    def __len__(self):
        return len(self.patient_ids)
    
    def __getitem__(self, idx):
        # Return all modalities as a dictionary
        sample = {
            'patient_id': self.patient_ids[idx]
        }
        
        for mod in self.modalities:
            patch = self.patches[mod][idx]
            sample[mod] = torch.FloatTensor(patch).unsqueeze(0)  # (1, D, H, W)
        
        return sample

In [ ]:
# Create dataset and dataloader
dataset = BraTS2020MultimodalDataset(
    data_root=DATA_ROOT,
    modalities=MODALITIES,
    patch_size=PATCH_SIZE,
    num_patches_per_volume=5
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

print(f"\nDataLoader created with {len(dataloader)} batches")

---
## 4. Deep Canonical Correlation Analysis (DCCA)
### 4.1 DCCA Architecture

In [ ]:
class DCCAEncoder(nn.Module):
    """3D CNN encoder for DCCA"""
    
    def __init__(self, input_channels=1, latent_dim=32):
        super().__init__()
        
        self.encoder = nn.Sequential(
            # Input: (1, 64, 64, 64)
            nn.Conv3d(input_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            # (32, 32, 32, 32)
            
            nn.Conv3d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            # (64, 16, 16, 16)
            
            nn.Conv3d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            # (128, 8, 8, 8)
            
            nn.Conv3d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm3d(256),
            nn.ReLU(inplace=True),
            # (256, 4, 4, 4)
            
            nn.AdaptiveAvgPool3d(1)
            # (256, 1, 1, 1)
        )
        
        self.fc = nn.Linear(256, latent_dim)
        
    def forward(self, x):
        features = self.encoder(x)
        features = features.view(features.size(0), -1)
        latent = self.fc(features)
        return latent

class DCCAModel(nn.Module):
    """DCCA model for learning correlated representations"""
    
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder1 = DCCAEncoder(input_channels=1, latent_dim=latent_dim)
        self.encoder2 = DCCAEncoder(input_channels=1, latent_dim=latent_dim)
        
    def forward(self, x1, x2):
        z1 = self.encoder1(x1)
        z2 = self.encoder2(x2)
        return z1, z2

### 4.2 DCCA Loss Function

In [ ]:
def dcca_loss(z1, z2, r1=1e-3, r2=1e-3):
    """
    Deep Canonical Correlation Analysis loss
    Maximizes correlation between two latent representations
    
    Args:
        z1: First latent representation (batch_size, latent_dim)
        z2: Second latent representation (batch_size, latent_dim)
        r1, r2: Regularization parameters
    """
    batch_size = z1.shape[0]
    latent_dim = z1.shape[1]
    
    # Center the data
    z1_mean = z1.mean(dim=0, keepdim=True)
    z2_mean = z2.mean(dim=0, keepdim=True)
    z1_centered = z1 - z1_mean
    z2_centered = z2 - z2_mean
    
    # Compute covariance matrices
    C11 = (z1_centered.t() @ z1_centered) / (batch_size - 1)
    C22 = (z2_centered.t() @ z2_centered) / (batch_size - 1)
    C12 = (z1_centered.t() @ z2_centered) / (batch_size - 1)
    
    # Add regularization
    C11 = C11 + r1 * torch.eye(latent_dim, device=z1.device)
    C22 = C22 + r2 * torch.eye(latent_dim, device=z2.device)
    
    # Compute correlation
    try:
        C11_inv_sqrt = torch.linalg.inv(torch.linalg.cholesky(C11))
        C22_inv_sqrt = torch.linalg.inv(torch.linalg.cholesky(C22))
        
        # Canonical correlation matrix
        T = C11_inv_sqrt @ C12 @ C22_inv_sqrt.t()
        
        # Total correlation (sum of singular values)
        corr = torch.trace(T @ T.t()).sqrt()
        
        # We want to maximize correlation, so minimize negative correlation
        loss = -corr
        
    except:
        # Fallback: use Frobenius norm of cross-covariance
        loss = -torch.norm(C12, p='fro')
    
    return loss

### 4.3 Train DCCA Models for All Modality Pairs

In [ ]:
# Train DCCA for each modality pair
dcca_models = {}
dcca_correlations = {}

print("\n" + "="*80)
print("TRAINING DCCA MODELS FOR ALL MODALITY PAIRS")
print("="*80)

for mod1, mod2 in MODALITY_PAIRS:
    pair_name = f"{MODALITY_NAMES[mod1]}-{MODALITY_NAMES[mod2]}"
    print(f"\n{'='*60}")
    print(f"Training DCCA: {pair_name}")
    print(f"{'='*60}")
    
    # Initialize model
    model = DCCAModel(latent_dim=LATENT_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # Training loop
    model.train()
    epoch_losses = []
    
    for epoch in range(NUM_EPOCHS_DCCA):
        total_loss = 0
        
        for batch in dataloader:
            x1 = batch[mod1].to(device)
            x2 = batch[mod2].to(device)
            
            optimizer.zero_grad()
            z1, z2 = model(x1, x2)
            loss = dcca_loss(z1, z2)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(dataloader)
        epoch_losses.append(avg_loss)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS_DCCA} - Loss: {avg_loss:.4f}")
    
    # Compute final correlation
    model.eval()
    all_z1 = []
    all_z2 = []
    
    with torch.no_grad():
        for batch in dataloader:
            x1 = batch[mod1].to(device)
            x2 = batch[mod2].to(device)
            z1, z2 = model(x1, x2)
            all_z1.append(z1.cpu())
            all_z2.append(z2.cpu())
    
    all_z1 = torch.cat(all_z1, dim=0)
    all_z2 = torch.cat(all_z2, dim=0)
    
    # Compute correlation
    correlation = torch.corrcoef(torch.cat([all_z1, all_z2], dim=1).t())
    mean_corr = correlation[:LATENT_DIM, LATENT_DIM:].abs().mean().item()
    
    dcca_models[(mod1, mod2)] = model
    dcca_correlations[(mod1, mod2)] = mean_corr
    
    print(f"Final DCCA correlation: {mean_corr:.4f}")

print("\n" + "="*80)
print("DCCA TRAINING COMPLETE")
print("="*80)

---
## 5. Hilbert-Schmidt Independence Criterion (HSIC)
### 5.1 HSIC Implementation

In [ ]:
def rbf_kernel(X, sigma=1.0):
    """
    Compute RBF (Gaussian) kernel matrix
    K(x, y) = exp(-||x - y||^2 / (2 * sigma^2))
    """
    # Compute pairwise squared distances
    XX = torch.sum(X**2, dim=1, keepdim=True)
    distances = XX + XX.t() - 2 * torch.mm(X, X.t())
    
    # Apply RBF kernel
    K = torch.exp(-distances / (2 * sigma**2))
    return K

def center_kernel_matrix(K):
    """Center a kernel matrix"""
    n = K.shape[0]
    H = torch.eye(n, device=K.device) - torch.ones(n, n, device=K.device) / n
    K_centered = H @ K @ H
    return K_centered

def hsic_score(X, Y, sigma_x=1.0, sigma_y=1.0):
    """
    Compute HSIC (Hilbert-Schmidt Independence Criterion)
    Measures statistical dependence between X and Y using kernel methods
    
    Args:
        X: First variable (n_samples, n_features)
        Y: Second variable (n_samples, n_features)
        sigma_x, sigma_y: Kernel bandwidth parameters
    
    Returns:
        HSIC score (higher = more dependent)
    """
    n = X.shape[0]
    
    # Compute kernel matrices
    K_x = rbf_kernel(X, sigma=sigma_x)
    K_y = rbf_kernel(Y, sigma=sigma_y)
    
    # Center kernel matrices
    K_x_centered = center_kernel_matrix(K_x)
    K_y_centered = center_kernel_matrix(K_y)
    
    # Compute HSIC
    hsic = torch.trace(K_x_centered @ K_y_centered) / ((n - 1) ** 2)
    
    return hsic.item()

### 5.2 Compute HSIC for All Modality Pairs

In [ ]:
# Compute HSIC for each modality pair
hsic_scores = {}

print("\n" + "="*80)
print("COMPUTING HSIC SCORES FOR ALL MODALITY PAIRS")
print("="*80)

# Extract features from all samples
all_features = {mod: [] for mod in MODALITIES}

print("\nExtracting features...")
for batch in tqdm(dataloader, desc="Processing batches"):
    for mod in MODALITIES:
        x = batch[mod].to(device)
        # Flatten to vector
        features = x.view(x.shape[0], -1)
        all_features[mod].append(features.cpu())

# Concatenate all features
for mod in MODALITIES:
    all_features[mod] = torch.cat(all_features[mod], dim=0)
    print(f"{MODALITY_NAMES[mod]} features: {all_features[mod].shape}")

# Compute HSIC for each pair
print("\nComputing HSIC scores...")
for mod1, mod2 in tqdm(MODALITY_PAIRS, desc="HSIC computation"):
    X = all_features[mod1]
    Y = all_features[mod2]
    
    # Use subset for efficiency (HSIC is O(n^2))
    n_samples = min(200, X.shape[0])
    indices = torch.randperm(X.shape[0])[:n_samples]
    X_subset = X[indices]
    Y_subset = Y[indices]
    
    # Compute median heuristic for bandwidth
    sigma_x = torch.median(torch.pdist(X_subset)).item()
    sigma_y = torch.median(torch.pdist(Y_subset)).item()
    
    # Compute HSIC
    hsic = hsic_score(X_subset, Y_subset, sigma_x=sigma_x, sigma_y=sigma_y)
    hsic_scores[(mod1, mod2)] = hsic
    
    pair_name = f"{MODALITY_NAMES[mod1]}-{MODALITY_NAMES[mod2]}"
    print(f"{pair_name}: HSIC = {hsic:.6f}")

print("\n" + "="*80)
print("HSIC COMPUTATION COMPLETE")
print("="*80)

---
## 6. Combined Redundancy Metrics

In [ ]:
# Normalize scores to [0, 1] range
dcca_values = np.array(list(dcca_correlations.values()))
hsic_values = np.array(list(hsic_scores.values()))

dcca_normalized = (dcca_values - dcca_values.min()) / (dcca_values.max() - dcca_values.min())
hsic_normalized = (hsic_values - hsic_values.min()) / (hsic_values.max() - hsic_values.min())

# Combined redundancy score (average of normalized DCCA and HSIC)
combined_redundancy = {}
for i, (mod1, mod2) in enumerate(MODALITY_PAIRS):
    combined = (dcca_normalized[i] + hsic_normalized[i]) / 2
    combined_redundancy[(mod1, mod2)] = combined

# Create summary dataframe
redundancy_df = pd.DataFrame([
    {
        'Modality_1': MODALITY_NAMES[mod1],
        'Modality_2': MODALITY_NAMES[mod2],
        'DCCA_Correlation': dcca_correlations[(mod1, mod2)],
        'HSIC_Score': hsic_scores[(mod1, mod2)],
        'Combined_Redundancy': combined_redundancy[(mod1, mod2)]
    }
    for mod1, mod2 in MODALITY_PAIRS
])

# Sort by combined redundancy
redundancy_df = redundancy_df.sort_values('Combined_Redundancy', ascending=False)

print("\n" + "="*80)
print("PROTOCOL REDUNDANCY ANALYSIS RESULTS")
print("="*80)
print(redundancy_df.to_string(index=False))
print("="*80)

# Save results
redundancy_df.to_csv(SCORES_DIR / 'redundancy_scores.csv', index=False)
print(f"\n✓ Saved: redundancy_scores.csv")

---
## 7. Advanced Visualizations
### 7.1 Redundancy Iceberg Plot

In [ ]:
# Redundancy Iceberg: Layered shared vs unique variance
fig, ax = plt.subplots(figsize=(12, 8))

# Sort pairs by redundancy
sorted_pairs = sorted(combined_redundancy.items(), key=lambda x: x[1], reverse=True)

# Colors for layers
colors = sns.color_palette("Blues_r", n_colors=len(sorted_pairs))

# Create layered bars (iceberg layers)
y_positions = np.arange(len(sorted_pairs))
layer_heights = []

for i, ((mod1, mod2), redundancy) in enumerate(sorted_pairs):
    # Shared variance (redundancy)
    shared = redundancy * 100
    # Unique variance (1 - redundancy)
    unique = (1 - redundancy) * 100
    
    # Draw layers
    ax.barh(y_positions[i], shared, left=0, height=0.7, 
            color=colors[i], edgecolor='white', linewidth=1.5,
            label=f'{MODALITY_NAMES[mod1]}-{MODALITY_NAMES[mod2]} (Shared)' if i < 3 else '')
    ax.barh(y_positions[i], unique, left=shared, height=0.7,
            color='lightgray', edgecolor='white', linewidth=1.5, alpha=0.5)
    
    layer_heights.append(shared)

# Add waterline to show "above" vs "below" water
waterline = np.median(layer_heights)
ax.axvline(waterline, color='steelblue', linewidth=3, linestyle='--', alpha=0.7, label='Median Redundancy')

# Labels
pair_labels = [f"{MODALITY_NAMES[mod1]}–{MODALITY_NAMES[mod2]}" for (mod1, mod2), _ in sorted_pairs]
ax.set_yticks(y_positions)
ax.set_yticklabels(pair_labels, fontsize=10)
ax.set_xlabel('Information Content (%)', fontsize=11)
ax.set_title('Redundancy Iceberg Plot\nShared vs Unique Information Between Modality Pairs', 
             fontsize=13, fontweight='bold', pad=15)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label='Shared Information (Redundant)'),
    Patch(facecolor='lightgray', alpha=0.5, label='Unique Information')
]
ax.legend(handles=legend_elements, loc='lower right', frameon=True, fancybox=False)

ax.set_xlim(0, 100)
ax.grid(axis='x', alpha=0.3)
sns.despine()

plt.tight_layout()
plt.savefig(VIZ_DIR / '01_redundancy_iceberg.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 01_redundancy_iceberg.png")

### 7.2 Modality Tetrahedron Map

In [ ]:
# Modality Tetrahedron: 4 modalities as vertices, edges = redundancy strength
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Tetrahedron vertices (4 modalities)
vertices = np.array([
    [0, 0, 0],           # FLAIR
    [1, 0, 0],           # T1
    [0.5, np.sqrt(3)/2, 0],  # T1CE
    [0.5, np.sqrt(3)/6, np.sqrt(2/3)]  # T2
])

modality_order = ['flair', 't1', 't1ce', 't2']
colors_mod = {'flair': '#3498db', 't1': '#e74c3c', 't1ce': '#2ecc71', 't2': '#f39c12'}

# Plot vertices
for i, mod in enumerate(modality_order):
    ax.scatter(*vertices[i], s=400, c=colors_mod[mod], 
               edgecolors='white', linewidths=2, alpha=0.9, depthshade=False)
    ax.text(vertices[i, 0], vertices[i, 1], vertices[i, 2] + 0.1, 
            MODALITY_NAMES[mod], fontsize=13, fontweight='bold', ha='center')

# Plot edges with thickness proportional to redundancy
for (mod1, mod2), redundancy in combined_redundancy.items():
    i = modality_order.index(mod1)
    j = modality_order.index(mod2)
    
    # Edge thickness based on redundancy
    linewidth = 1 + redundancy * 8  # Scale to 1-9
    
    # Draw edge
    ax.plot([vertices[i, 0], vertices[j, 0]],
            [vertices[i, 1], vertices[j, 1]],
            [vertices[i, 2], vertices[j, 2]],
            color='gray', linewidth=linewidth, alpha=0.6)
    
    # Add redundancy score at edge midpoint
    mid = (vertices[i] + vertices[j]) / 2
    ax.text(mid[0], mid[1], mid[2], f'{redundancy:.2f}',
            fontsize=8, ha='center', bbox=dict(boxstyle='round,pad=0.3', 
            facecolor='white', edgecolor='gray', alpha=0.8))

# Styling
ax.set_xlabel('X', fontsize=10)
ax.set_ylabel('Y', fontsize=10)
ax.set_zlabel('Z', fontsize=10)
ax.set_title('Modality Tetrahedron Map\nEdge Thickness = Redundancy Strength', 
             fontsize=13, fontweight='bold', pad=20)
ax.view_init(elev=20, azim=45)

# Remove grid for cleaner look
ax.grid(False)
ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False

plt.tight_layout()
plt.savefig(VIZ_DIR / '02_modality_tetrahedron.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 02_modality_tetrahedron.png")

### 7.3 Curl-Field Vector Map

In [ ]:
# Curl-Field Vector Map: Redundancy gradients inside brain volume
# Take a sample volume and compute local redundancy

# Get a sample subject
sample_idx = 0
sample_data = dataset[sample_idx]

# Compute local redundancy on a 2D slice
slice_idx = PATCH_SIZE // 2
grid_size = 8  # Divide into 8x8 grid
step = PATCH_SIZE // grid_size

# Create grid for vector field
redundancy_grid = np.zeros((grid_size, grid_size))

# Compute redundancy for each grid cell using HSIC on local patches
for i in range(grid_size):
    for j in range(grid_size):
        # Extract local patch
        x_start, y_start = i * step, j * step
        x_end, y_end = x_start + step, y_start + step
        
        # Average redundancy across all pairs for this location
        local_redundancies = []
        for mod1, mod2 in MODALITY_PAIRS:
            patch1 = sample_data[mod1][0, x_start:x_end, y_start:y_end, slice_idx]
            patch2 = sample_data[mod2][0, x_start:x_end, y_start:y_end, slice_idx]
            
            # Simple correlation as proxy for local redundancy
            corr = np.corrcoef(patch1.flatten(), patch2.flatten())[0, 1]
            local_redundancies.append(abs(corr))
        
        redundancy_grid[i, j] = np.mean(local_redundancies)

# Compute gradient (curl field)
grad_y, grad_x = np.gradient(redundancy_grid)

# Create visualization
fig, ax = plt.subplots(figsize=(12, 10))

# Background: redundancy heatmap
im = ax.imshow(redundancy_grid.T, cmap='RdYlBu_r', origin='lower', 
               extent=[0, PATCH_SIZE, 0, PATCH_SIZE], alpha=0.7)

# Overlay: vector field (curl)
X, Y = np.meshgrid(np.linspace(step/2, PATCH_SIZE-step/2, grid_size),
                   np.linspace(step/2, PATCH_SIZE-step/2, grid_size))

# Normalize vectors for better visualization
magnitude = np.sqrt(grad_x**2 + grad_y**2)
grad_x_norm = grad_x / (magnitude + 1e-6)
grad_y_norm = grad_y / (magnitude + 1e-6)

ax.quiver(X, Y, grad_x_norm.T, grad_y_norm.T, magnitude.T,
          cmap='viridis', scale=15, width=0.004, alpha=0.8,
          headwidth=4, headlength=5)

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Local Redundancy', fontsize=11)

ax.set_xlabel('X Position (voxels)', fontsize=11)
ax.set_ylabel('Y Position (voxels)', fontsize=11)
ax.set_title('Curl-Field Vector Map\nRedundancy Gradient Flow in Brain Volume', 
             fontsize=13, fontweight='bold', pad=15)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(VIZ_DIR / '03_curl_field_vector_map.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 03_curl_field_vector_map.png")

### 7.4 Chord-Galaxy Graph

In [ ]:
# Chord-Galaxy Graph: Modality-to-modality redundancy as glowing chords
fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(projection='polar'))

# Position modalities around circle
n_modalities = len(MODALITIES)
angles = np.linspace(0, 2 * np.pi, n_modalities, endpoint=False)

modality_positions = {mod: angles[i] for i, mod in enumerate(MODALITIES)}
colors_chord = {'flair': '#3498db', 't1': '#e74c3c', 't1ce': '#2ecc71', 't2': '#f39c12'}

# Draw modality nodes
for mod, angle in modality_positions.items():
    ax.scatter(angle, 1.0, s=800, c=colors_chord[mod], 
               edgecolors='white', linewidths=3, alpha=0.9, zorder=10)
    
    # Add labels outside the circle
    label_angle = angle
    ha = 'left' if 0 <= label_angle <= np.pi else 'right'
    ax.text(label_angle, 1.15, MODALITY_NAMES[mod], 
            fontsize=14, fontweight='bold', ha=ha, va='center')

# Draw chords (curves connecting modalities)
for (mod1, mod2), redundancy in combined_redundancy.items():
    angle1 = modality_positions[mod1]
    angle2 = modality_positions[mod2]
    
    # Create bezier curve for chord
    theta = np.linspace(angle1, angle2, 100)
    
    # Radius varies to create curve (lower in middle)
    r = 1.0 - 0.5 * redundancy * np.sin(np.linspace(0, np.pi, 100))
    
    # Color based on redundancy
    color = plt.cm.plasma(redundancy)
    linewidth = 2 + redundancy * 6
    
    ax.plot(theta, r, color=color, linewidth=linewidth, alpha=0.7, zorder=5)

# Styling
ax.set_ylim(0, 1.2)
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.axis('off')
ax.set_title('Chord-Galaxy Graph\nRedundancy Network Between MRI Modalities', 
             fontsize=14, fontweight='bold', pad=30, y=1.05)

# Add colorbar for redundancy
sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.08, orientation='horizontal')
cbar.set_label('Redundancy Strength', fontsize=11)

plt.tight_layout()
plt.savefig(VIZ_DIR / '04_chord_galaxy_graph.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 04_chord_galaxy_graph.png")

### 7.5 Redundancy Jigsaw Map

In [ ]:
# Redundancy Jigsaw: Divide brain into blocks colored by local redundancy
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# For each modality pair, show local redundancy map
for idx, (mod1, mod2) in enumerate(MODALITY_PAIRS):
    ax = axes[idx]
    
    # Compute local redundancy grid
    grid_size = 16
    step = PATCH_SIZE // grid_size
    local_redundancy = np.zeros((grid_size, grid_size))
    
    for i in range(grid_size):
        for j in range(grid_size):
            x_start, y_start = i * step, j * step
            x_end, y_end = x_start + step, y_start + step
            
            patch1 = sample_data[mod1][0, x_start:x_end, y_start:y_end, slice_idx]
            patch2 = sample_data[mod2][0, x_start:x_end, y_start:y_end, slice_idx]
            
            # Correlation as local redundancy
            if patch1.std() > 0.01 and patch2.std() > 0.01:
                corr = np.corrcoef(patch1.flatten(), patch2.flatten())[0, 1]
                local_redundancy[i, j] = abs(corr)
    
    # Plot jigsaw (discrete blocks)
    im = ax.imshow(local_redundancy.T, cmap='YlOrRd', origin='lower',
                   extent=[0, PATCH_SIZE, 0, PATCH_SIZE], 
                   vmin=0, vmax=1, interpolation='nearest')
    
    # Add grid lines for jigsaw effect
    for i in range(grid_size + 1):
        ax.axhline(i * step, color='white', linewidth=1, alpha=0.5)
        ax.axvline(i * step, color='white', linewidth=1, alpha=0.5)
    
    ax.set_title(f'{MODALITY_NAMES[mod1]} ↔ {MODALITY_NAMES[mod2]}', 
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('X', fontsize=9)
    ax.set_ylabel('Y', fontsize=9)
    ax.set_aspect('equal')
    
    # Colorbar
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Redundancy Jigsaw Map\nLocal Block-wise Redundancy Patterns', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(VIZ_DIR / '05_redundancy_jigsaw_map.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: 05_redundancy_jigsaw_map.png")

---
## 8. Save Complete Results

In [ ]:
# Create redundancy matrices
redundancy_matrix_dcca = np.zeros((len(MODALITIES), len(MODALITIES)))
redundancy_matrix_hsic = np.zeros((len(MODALITIES), len(MODALITIES)))
redundancy_matrix_combined = np.zeros((len(MODALITIES), len(MODALITIES)))

for (mod1, mod2), dcca_corr in dcca_correlations.items():
    i = MODALITIES.index(mod1)
    j = MODALITIES.index(mod2)
    
    redundancy_matrix_dcca[i, j] = dcca_corr
    redundancy_matrix_dcca[j, i] = dcca_corr
    
    redundancy_matrix_hsic[i, j] = hsic_scores[(mod1, mod2)]
    redundancy_matrix_hsic[j, i] = hsic_scores[(mod1, mod2)]
    
    redundancy_matrix_combined[i, j] = combined_redundancy[(mod1, mod2)]
    redundancy_matrix_combined[j, i] = combined_redundancy[(mod1, mod2)]

# Set diagonal to 1 (self-redundancy)
np.fill_diagonal(redundancy_matrix_dcca, 1.0)
np.fill_diagonal(redundancy_matrix_hsic, 1.0)
np.fill_diagonal(redundancy_matrix_combined, 1.0)

# Save matrices
modality_labels = [MODALITY_NAMES[mod] for mod in MODALITIES]

pd.DataFrame(redundancy_matrix_dcca, 
             index=modality_labels, 
             columns=modality_labels).to_csv(SCORES_DIR / 'redundancy_matrix_dcca.csv')

pd.DataFrame(redundancy_matrix_hsic, 
             index=modality_labels, 
             columns=modality_labels).to_csv(SCORES_DIR / 'redundancy_matrix_hsic.csv')

pd.DataFrame(redundancy_matrix_combined, 
             index=modality_labels, 
             columns=modality_labels).to_csv(SCORES_DIR / 'redundancy_matrix_combined.csv')

print("\n✓ Saved redundancy matrices")
print("  - redundancy_matrix_dcca.csv")
print("  - redundancy_matrix_hsic.csv")
print("  - redundancy_matrix_combined.csv")

---
## 9. Interpretation and Analysis

In [ ]:
print("\n" + "="*80)
print("PROTOCOL REDUNDANCY ANALYSIS - INTERPRETATION")
print("="*80)

# Find most and least redundant pairs
most_redundant = redundancy_df.iloc[0]
least_redundant = redundancy_df.iloc[-1]

print("\n📊 KEY FINDINGS:\n")

print("1. MOST REDUNDANT PAIR:")
print(f"   {most_redundant['Modality_1']} ↔ {most_redundant['Modality_2']}")
print(f"   Combined Redundancy: {most_redundant['Combined_Redundancy']:.3f}")
print(f"   DCCA Correlation: {most_redundant['DCCA_Correlation']:.3f}")
print(f"   HSIC Score: {most_redundant['HSIC_Score']:.6f}")
print("   → High redundancy suggests these modalities share substantial information")
print("   → May be candidates for protocol reduction in resource-limited settings\n")

print("2. LEAST REDUNDANT PAIR:")
print(f"   {least_redundant['Modality_1']} ↔ {least_redundant['Modality_2']}")
print(f"   Combined Redundancy: {least_redundant['Combined_Redundancy']:.3f}")
print(f"   DCCA Correlation: {least_redundant['DCCA_Correlation']:.3f}")
print(f"   HSIC Score: {least_redundant['HSIC_Score']:.6f}")
print("   → Low redundancy indicates complementary, unique information")
print("   → Both modalities are essential for comprehensive imaging\n")

print("3. OVERALL REDUNDANCY PATTERNS:")
print(f"   Mean redundancy: {redundancy_df['Combined_Redundancy'].mean():.3f}")
print(f"   Std redundancy: {redundancy_df['Combined_Redundancy'].std():.3f}")
print(f"   Range: [{redundancy_df['Combined_Redundancy'].min():.3f}, "
      f"{redundancy_df['Combined_Redundancy'].max():.3f}]\n")

print("4. MRI PHYSICS INTERPRETATION:\n")

# T1 vs T1CE
t1_t1ce = redundancy_df[(redundancy_df['Modality_1'].isin(['T1', 'T1CE'])) & 
                        (redundancy_df['Modality_2'].isin(['T1', 'T1CE']))]
if not t1_t1ce.empty:
    print(f"   T1 ↔ T1CE: Redundancy = {t1_t1ce['Combined_Redundancy'].values[0]:.3f}")
    print("   → T1CE is T1 with contrast enhancement (gadolinium)")
    print("   → High redundancy expected due to shared T1 relaxation physics")
    print("   → T1CE adds unique vascular/BBB permeability information\n")

# T2 vs FLAIR
t2_flair = redundancy_df[(redundancy_df['Modality_1'].isin(['T2', 'FLAIR'])) & 
                         (redundancy_df['Modality_2'].isin(['T2', 'FLAIR']))]
if not t2_flair.empty:
    print(f"   T2 ↔ FLAIR: Redundancy = {t2_flair['Combined_Redundancy'].values[0]:.3f}")
    print("   → FLAIR is T2-weighted with fluid suppression")
    print("   → Moderate-high redundancy expected")
    print("   → FLAIR adds unique periventricular lesion detection\n")

# T1 vs T2
t1_t2 = redundancy_df[(redundancy_df['Modality_1'].isin(['T1', 'T2'])) & 
                      (redundancy_df['Modality_2'].isin(['T1', 'T2']))]
if not t1_t2.empty:
    print(f"   T1 ↔ T2: Redundancy = {t1_t2['Combined_Redundancy'].values[0]:.3f}")
    print("   → Fundamentally different relaxation mechanisms")
    print("   → Low redundancy expected - highly complementary")
    print("   → T1: anatomy/structure, T2: pathology/edema\n")

print("5. CLINICAL IMPLICATIONS:\n")
print("   • High redundancy pairs: Consider acquisition time optimization")
print("   • Low redundancy pairs: Essential for diagnostic completeness")
print("   • DCCA reveals nonlinear shared representations beyond simple correlation")
print("   • HSIC captures statistical dependencies missed by linear methods\n")

print("="*80)
print("ANALYSIS COMPLETE")
print("="*80)

---
## 10. Summary and Output Manifest

In [ ]:
print("\n" + "="*80)
print("PROTOCOL REDUNDANCY ANALYSIS - COMPLETE")
print("="*80)

print("\n📁 OUTPUT DIRECTORIES:")
print(f"   Visualizations: {VIZ_DIR.resolve()}")
print(f"   Scores: {SCORES_DIR.resolve()}")

print("\n🎨 GENERATED VISUALIZATIONS:")
print("   1. Redundancy Iceberg Plot")
print("   2. Modality Tetrahedron Map")
print("   3. Curl-Field Vector Map")
print("   4. Chord-Galaxy Graph")
print("   5. Redundancy Jigsaw Map")

print("\n📊 SAVED DATA FILES:")
print("   • redundancy_scores.csv - Complete pairwise scores")
print("   • redundancy_matrix_dcca.csv - DCCA correlation matrix")
print("   • redundancy_matrix_hsic.csv - HSIC score matrix")
print("   • redundancy_matrix_combined.csv - Combined redundancy matrix")

print("\n🔬 METHODS USED:")
print("   • DCCA: Deep Canonical Correlation Analysis (nonlinear)")
print("   • HSIC: Hilbert-Schmidt Independence Criterion (kernel-based)")
print("   • Combined metric: Normalized average of DCCA + HSIC")

print("\n✅ NOTEBOOK READY FOR:")
print("   • Research publication")
print("   • Protocol optimization studies")
print("   • Clinical workflow analysis")
print("   • Resource allocation planning")

print("\n" + "="*80)
print("Thank you for using Protocol Redundancy Analysis!")
print("="*80)